In [0]:
%python

import os, sys
current_directory = os.getcwd()
parent_directory = os.path.dirname(current_directory)

# Read Functions folder relative to current notebook
sys.path.append(parent_directory + '/Functions')

from helper_functions import read_csv
from transform_functions import create_missing_records, add_anomaly, apply_schema
from schema_definitions import silver_schema, gold_schema
from pyspark.sql.functions import col, current_timestamp, min, max, avg, lit

# Get DLT Parameter
path = spark.conf.get("path")

In [0]:
@dlt.table(
    name=f"wind_turbines_raw",
    comment="table to store raw bronze data",
    partition_cols=["turbine_id"]
)
@dlt.expect_or_fail("valid_wind_direction", "wind_direction BETWEEN 0 AND 359")
def wind_turbines_raw():
    # read raw data and adds metadata columns
    new_bronze_df = read_csv(f"{path}")
    
    return new_bronze_df

In [0]:
@dlt.table(
    name=f"wind_turbines_cleansed",
    comment=f"Table to store cleansed silver layer",
    partition_cols=["turbine_id", "date"]
)
def wind_turbines_cleansed():
    bronze_df = spark.read.table("wind_turbines_raw")

    # create records if missing from raw files
    silver_df = create_missing_records(bronze_df)

    # Add anomaly column
    silver_df = add_anomaly(silver_df)

    # apply schema definition to final dataframe
    silver_df = apply_schema(silver_df, silver_schema)
    
    return silver_df

In [0]:
@dlt.table(
    name=f"wind_turbines_aggregated",
    comment=f"Table to store aggregated gold layer",
    partition_cols=["turbine_id", "date"]
)
def wind_turbines_aggregated():
    silver_df = spark.read.table("wind_turbines_cleansed")
    
    # filter out bad records before producing summary
    gold_df = (
                silver_df
                    .filter((col("anomaly") == False) | (col("input_file_name") != "UNDEF"))
                    .groupBy("date", "turbine_id")
                    .agg(min("power_output").alias("min_power_output"),
                        max("power_output").alias("max_power_output"),
                        avg("power_output").alias("avg_power_output"))
                )

    # apply schema definition to final dataframe
    gold_df = apply_schema(gold_df, gold_schema)

    return gold_df